# 📊 Analisis UMKM Indonesia
## Menggunakan Dataset REAL dari Kaggle

**Dataset**: Indonesia E-Commerce Sales & Shipping 2023-2025

**Cara Penggunaan:**
1. Upload `kaggle.json` (dari https://kaggle.com/settings → API → Create Token)
2. Klik `Runtime` → `Run all`
3. Download hasil Excel

---

In [ ]:
# SETUP & INSTALL
!pip install kaggle plotly openpyxl --quiet

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, silhouette_score

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

print("✅ Libraries loaded!")
print(f"📅 Waktu: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📥 Upload Kaggle Credentials & Download Dataset

In [ ]:
# Upload kaggle.json dari komputer Anda
from google.colab import files

print("📤 Upload file kaggle.json dari komputer Anda:")
print("   (Dapatkan dari: https://kaggle.com/settings → API → Create New Token)")
print()

try:
    uploaded = files.upload()
    
    # Setup kaggle credentials
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    print("\n✅ Kaggle credentials configured!")
    KAGGLE_AVAILABLE = True
except:
    print("⚠️ Upload dibatalkan. Akan menggunakan data simulasi.")
    KAGGLE_AVAILABLE = False

In [ ]:
# Download dataset dari Kaggle
DATA_SOURCE = "Simulated Data"

if KAGGLE_AVAILABLE:
    print("📥 Downloading Indonesia E-Commerce dataset from Kaggle...")
    !kaggle datasets download -d bakitacos/indonesia-ecommerce-sales-shipping-20232025 --unzip -p kaggle_data
    
    # Find CSV file
    import os
    csv_files = [f for f in os.listdir('kaggle_data') if f.endswith('.csv')]
    if csv_files:
        # Prefer 'clean' version
        clean_files = [f for f in csv_files if 'clean' in f.lower()]
        kaggle_file = f"kaggle_data/{clean_files[0] if clean_files else csv_files[0]}"
        print(f"✅ Found: {kaggle_file}")
        
        df = pd.read_csv(kaggle_file)
        df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
        DATA_SOURCE = "Kaggle: Indonesia E-Commerce 2023-2025"
        print(f"✅ Loaded {len(df):,} records from Kaggle")
    else:
        print("⚠️ No CSV found, using simulated data")
        KAGGLE_AVAILABLE = False

In [ ]:
# Fallback: Generate simulated data if Kaggle not available
if not KAGGLE_AVAILABLE or 'df' not in dir():
    print("📊 Generating simulated UMKM data...")
    
    categories = ['Makanan & Minuman', 'Pakaian', 'Kerajinan', 'Elektronik', 'Pertanian', 'Jasa', 'Kosmetik', 'Furniture']
    regions = ['Jakarta', 'Jawa Barat', 'Jawa Tengah', 'Jawa Timur', 'Bali', 'Sumatera', 'Sulawesi', 'Kalimantan']
    products = ['Kopi Arabika', 'Batik', 'Tas Rotan', 'Power Bank', 'Beras Organik', 'Laundry', 'Lulur', 'Kursi Rotan'] * 8
    
    data = []
    np.random.seed(42)
    for i in range(5000):
        cat = np.random.choice(categories)
        base_price = 300000 if cat in ['Elektronik', 'Furniture'] else 80000
        data.append({
            'product_id': f'PROD_{i//50:04d}',
            'product_name': np.random.choice(products),
            'category': cat,
            'price': round(base_price * np.random.uniform(0.5, 1.5), -2),
            'sales_count': np.random.randint(5, 50),
            'discount_percent': np.random.choice([0, 5, 10, 15], p=[0.6, 0.2, 0.15, 0.05]),
            'rating': round(np.random.uniform(3.5, 5.0), 1),
            'sale_date': (datetime.now() - timedelta(days=np.random.randint(0, 90))).strftime('%Y-%m-%d'),
            'region': np.random.choice(regions),
            'seller_id': f'SELLER_{np.random.randint(1,50):03d}'
        })
    df = pd.DataFrame(data)
    DATA_SOURCE = "Simulated UMKM Data"
    print(f"✅ Generated {len(df):,} records")

In [ ]:
# Preprocess data
print(f"📊 Data Source: {DATA_SOURCE}")
print(f"Columns: {list(df.columns)}")

# Ensure required columns exist
if 'revenue' not in df.columns:
    df['price'] = pd.to_numeric(df.get('price', df.get('unit_price', df.get('item_price', 50000))), errors='coerce').fillna(50000)
    df['sales_count'] = pd.to_numeric(df.get('sales_count', df.get('quantity', df.get('qty', 1))), errors='coerce').fillna(1).astype(int)
    df['discount_percent'] = pd.to_numeric(df.get('discount_percent', df.get('discount', 0)), errors='coerce').fillna(0)
    df['revenue'] = df['price'] * df['sales_count'] * (1 - df['discount_percent']/100)

if 'sale_date' not in df.columns:
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    if date_cols:
        df['sale_date'] = df[date_cols[0]]
    else:
        df['sale_date'] = pd.date_range(end=datetime.now(), periods=len(df)).strftime('%Y-%m-%d')

df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce')
df['day_of_week'] = df['sale_date'].dt.dayofweek
df['month'] = df['sale_date'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print(f"\n✅ Preprocessed. Shape: {df.shape}")
df.head()

## 📈 Exploratory Data Analysis

In [ ]:
print("📊 Statistik Deskriptif:")
display(df[['price', 'sales_count', 'discount_percent', 'revenue']].describe().round(2))

if 'category' in df.columns:
    category_stats = df.groupby('category').agg({'sales_count': 'sum', 'revenue': 'sum', 'price': 'mean'}).round(2)
    category_stats.columns = ['Total Sales', 'Total Revenue', 'Avg Price']
    print("\n📦 Per Kategori:")
    display(category_stats.sort_values('Total Revenue', ascending=False))

## 📊 Visualizations

In [ ]:
# Sales over time
daily = df.groupby(df['sale_date'].dt.date).agg({'sales_count': 'sum', 'revenue': 'sum'}).reset_index()
fig = px.line(daily, x='sale_date', y='sales_count', title='📈 Daily Sales Trend')
fig.show()

In [ ]:
# Revenue by category
if 'category' in df.columns:
    fig = px.pie(df.groupby('category')['revenue'].sum().reset_index(), values='revenue', names='category', title='💰 Revenue by Category')
    fig.show()

## 🤖 Machine Learning

In [ ]:
# Prepare features
ml = df.copy()
ml['category_encoded'] = LabelEncoder().fit_transform(ml['category'].astype(str)) if 'category' in ml.columns else 0
ml['sales_lag_1'] = ml['sales_count'].shift(1).fillna(ml['sales_count'].mean())
ml['sales_ma_7'] = ml['sales_count'].rolling(7, min_periods=1).mean()

features = ['price', 'discount_percent', 'day_of_week', 'month', 'is_weekend', 'sales_lag_1', 'sales_ma_7', 'category_encoded']
X = ml[features].fillna(0)
y = ml['sales_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train models
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42).fit(X_train, y_train)
print(f"🔹 Gradient Boosting R²: {r2_score(y_test, gb.predict(X_test)):.4f}")

# Feature importance
imp = pd.DataFrame({'feature': features, 'importance': gb.feature_importances_}).sort_values('importance', ascending=False)
fig = px.bar(imp, x='importance', y='feature', orientation='h', title='📊 Feature Importance')
fig.show()

In [ ]:
# Product Clustering
prod = df.groupby('product_id' if 'product_id' in df.columns else df.index).agg({'price': 'mean', 'sales_count': 'sum', 'revenue': 'sum'}).reset_index()
if len(prod) >= 5:
    scaled = StandardScaler().fit_transform(prod[['price', 'sales_count', 'revenue']])
    prod['cluster'] = KMeans(n_clusters=min(5, len(prod)), random_state=42, n_init=10).fit_predict(scaled)
    prod['segment'] = prod['cluster'].map({0:'Standard', 1:'Premium', 2:'Budget', 3:'Best Sellers', 4:'Low Performers'})
    fig = px.scatter(prod, x='price', y='sales_count', color='segment', size='revenue', title='🎯 Product Segments')
    fig.show()
product_features = prod

## 🔮 7-Day Predictions

In [ ]:
# Predict next 7 days
predictions = []
for day in range(1, 8):
    future = datetime.now() + timedelta(days=day)
    pred_X = pd.DataFrame([{
        'price': df['price'].mean(),
        'discount_percent': df['discount_percent'].mean(),
        'day_of_week': future.weekday(),
        'month': future.month,
        'is_weekend': 1 if future.weekday() >= 5 else 0,
        'sales_lag_1': df['sales_count'].mean(),
        'sales_ma_7': df['sales_count'].mean(),
        'category_encoded': 0
    }])
    pred = gb.predict(pred_X)[0] * len(df['product_id'].unique()) if 'product_id' in df.columns else gb.predict(pred_X)[0] * 50
    predictions.append({'Tanggal': future.strftime('%Y-%m-%d'), 'Hari': future.strftime('%A'), 'Prediksi_Sales': int(pred), 'Prediksi_Revenue': int(pred * df['price'].mean() * 0.9)})

predictions_df = pd.DataFrame(predictions)
display(predictions_df)

## 💾 Export (Format Rapih)

In [ ]:
# Format dan export
df_exp = df.copy()
df_exp['price_formatted'] = df_exp['price'].apply(lambda x: f"Rp {x:,.0f}".replace(',','.'))
df_exp['revenue_formatted'] = df_exp['revenue'].apply(lambda x: f"Rp {x:,.0f}".replace(',','.'))

pred_exp = predictions_df.copy()
pred_exp['Revenue_Formatted'] = pred_exp['Prediksi_Revenue'].apply(lambda x: f"Rp {x:,.0f}".replace(',','.'))

# CSV dengan semicolon
df_exp.to_csv('umkm_full_data.csv', index=False, sep=';', encoding='utf-8-sig')
product_features.to_csv('umkm_product_segments.csv', index=False, sep=';', encoding='utf-8-sig')
pred_exp.to_csv('umkm_predictions.csv', index=False, sep=';', encoding='utf-8-sig')

# Excel
with pd.ExcelWriter('umkm_analytics_results.xlsx', engine='openpyxl') as w:
    df.head(1000).to_excel(w, sheet_name='Data Penjualan', index=False)
    product_features.to_excel(w, sheet_name='Segmentasi Produk', index=False)
    predictions_df.to_excel(w, sheet_name='Prediksi 7 Hari', index=False)

print(f"""\n✅ Export selesai!
📊 Data Source: {DATA_SOURCE}
📁 Files:
   ⭐ umkm_analytics_results.xlsx (REKOMENDASI)
   • umkm_full_data.csv (separator: ;)
   • umkm_product_segments.csv
   • umkm_predictions.csv
""")

# Download
from google.colab import files
files.download('umkm_analytics_results.xlsx')